# Exercise 3 — score_headlines

`score_headlines` is the batch version: map `score_headline` over a list of headlines. In practice, a trading day produces multiple news items — earnings releases, central bank statements, economic data. Scoring all of them and aggregating gives a more robust daily signal than relying on a single headline.

In [ ]:
import re

# Gate-safe mock LLM — keyword-based, deterministic, no Ollama required
# Checks only the user message to avoid matching keywords in the system prompt.
def _mock_llm(messages):
    user_text = next(
        (m.get("content", "") for m in messages if m.get("role") == "user"), ""
    ).lower()
    if any(w in user_text for w in ["surge", "rally", "rise", "gain", "bull", "strong"]):
        return "0.75"
    if any(w in user_text for w in ["crash", "fall", "decline", "bear", "weak", "loss"]):
        return "-0.60"
    return "0.10"

BULLISH_HEADLINES = [
    "Tech stocks rally on strong earnings",
    "Markets surge as Fed signals rate pause",
    "S&P 500 gains 2% on positive jobs data",
    "Bull market continues with broad gains",
]
BEARISH_HEADLINES = [
    "Markets crash amid recession fears",
    "Stocks fall sharply on weak economic data",
    "S&P 500 declines on hawkish Fed remarks",
    "Bear market deepens as losses mount",
]
NEUTRAL_HEADLINES = [
    "Markets trade sideways in quiet session",
    "Mixed signals leave investors cautious",
    "Stocks finish flat as investors await data",
]
def parse_score(text):
    """Extract and clamp a float from LLM output. Returns 0.0 if not found."""
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches:
        return 0.0
    return max(-1.0, min(1.0, float(matches[0])))

def build_sentiment_prompt(headline):
    return [
        {
            "role": "system",
            "content": (
                "You are a financial news sentiment analyzer. "
                "Score the sentiment from -1.0 (very bearish) to 1.0 (very bullish). "
                "Reply with ONLY a single decimal number. No explanation."
            ),
        },
        {"role": "user", "content": f"Headline: {headline}"},
    ]
def score_headline(headline, llm_fn=None):
    messages = build_sentiment_prompt(headline)
    if llm_fn is not None:
        response = llm_fn(messages)
    else:
        import ollama
        response = ollama.chat(model="llama3.2", messages=messages)["message"]["content"]
    return parse_score(response)

def score_headlines(headlines, llm_fn=None):
    """Score a list of headlines.

    Args:
        headlines : list[str]
        llm_fn    : optional injection (same signature as score_headline)

    Returns:
        list[float] — same length as headlines; each value in [-1.0, 1.0].

    Implementation:
        return [score_headline(h, llm_fn) for h in headlines]
    """
    # TODO: one line
    return [0.0] * len(headlines)


### Checks

In [ ]:
checks = 0

# 1 — returns a list of the same length as input
try:
    scores = score_headlines(BULLISH_HEADLINES, llm_fn=_mock_llm)
    assert isinstance(scores, list) and len(scores) == len(BULLISH_HEADLINES)
    checks += 1; print("✅ 1 returns list of same length as input")
except Exception as e:
    print("❌ 1:", e)

# 2 — all values are floats in [-1.0, 1.0]
try:
    scores = score_headlines(BULLISH_HEADLINES + BEARISH_HEADLINES, llm_fn=_mock_llm)
    for s in scores:
        assert isinstance(s, float) and -1.0 <= s <= 1.0, f"invalid score: {s}"
    checks += 1; print("✅ 2 all values are float in [-1.0, 1.0]")
except Exception as e:
    print("❌ 2:", e)

# 3 — bullish headlines → all positive scores with mock
try:
    scores = score_headlines(BULLISH_HEADLINES, llm_fn=_mock_llm)
    assert all(s > 0 for s in scores), f"expected all positive: {scores}"
    checks += 1; print("✅ 3 bullish headlines → all positive scores")
except Exception as e:
    print("❌ 3:", e)

# 4 — bearish headlines → all negative scores with mock
try:
    scores = score_headlines(BEARISH_HEADLINES, llm_fn=_mock_llm)
    assert all(s < 0 for s in scores), f"expected all negative: {scores}"
    checks += 1; print("✅ 4 bearish headlines → all negative scores")
except Exception as e:
    print("❌ 4:", e)

# 5 — empty list returns empty list
try:
    scores = score_headlines([], llm_fn=_mock_llm)
    assert scores == [], f"expected [], got {scores}"
    checks += 1; print("✅ 5 empty headlines list → empty scores list")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
